#Generating my first text

1. Loading the model and tokenizer

  These are the different available good models from hugging face,
*   LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
*   PHI = "microsoft/Phi-4-mini-instruct"
*   GEMMA = "google/gemma-3-270m-it"
*   QWEN = "Qwen/Qwen3-4B-Instruct-2507"
*   DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"



*   ##Inorder to use these models you need to have a hugging face login token



In [1]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('Murali')
login(hf_token, add_to_git_credential=True)

In [2]:
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [64]:
model = AutoModelForCausalLM.from_pretrained(GEMMA,device_map = "auto", torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(GEMMA)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

##Now we create a pipeline, so that we have the model, tokenizer and text generation process in a single function

In [7]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False, # this means the prompt will not be returned. Just the output is returned.
    max_new_tokens=500, # to avoid unecessary lengthy text generation
    do_sample = False # picks only most probable next word, instead of sampling between next probable tokens
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#Now we create a prompt or query that we wish to ask the model

In [22]:
messages = [{"role":"system", "content":"you are a helpful assistant"},
            {"role":"user", "content": "what is llama?"}]

#Now we call the pipeline/generate function with our messages

In [23]:
output = generator(messages)


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [24]:
print(output) # this will give response as list of dictionaries

[{'generated_text': 'Llama is a term that can refer to several different things, depending on the context:\n\n1. **Animal**: Llama (Lama glama) is a domesticated South American camelid, native to the high plains of the Andes in countries like Peru, Bolivia, Ecuador, and Chile. They are known for their long necks, thick fur, and are often used as pack animals or for their wool.\n\n2. **AI Model**: Llama can also refer to a type of large language model developed by Microsoft, known as Microsoft\'s Phi language model. This model is designed to understand and generate human-like text, and it is part of Microsoft\'s efforts in advancing AI capabilities.\n\n3. **Other Uses**: In some contexts, "Llama" might be used as a nickname or a term in various other fields or communities, but these are less common.\n\nIf you have a specific context in mind, please provide more details so I can give a more precise answer.'}]


In [25]:
generated_text = output[0]['generated_text'] # we access only the response this way
print(generated_text)

Llama is a term that can refer to several different things, depending on the context:

1. **Animal**: Llama (Lama glama) is a domesticated South American camelid, native to the high plains of the Andes in countries like Peru, Bolivia, Ecuador, and Chile. They are known for their long necks, thick fur, and are often used as pack animals or for their wool.

2. **AI Model**: Llama can also refer to a type of large language model developed by Microsoft, known as Microsoft's Phi language model. This model is designed to understand and generate human-like text, and it is part of Microsoft's efforts in advancing AI capabilities.

3. **Other Uses**: In some contexts, "Llama" might be used as a nickname or a term in various other fields or communities, but these are less common.

If you have a specific context in mind, please provide more details so I can give a more precise answer.


#Now we see tokenization.


*   we first declare our prompt
*   then we tokenize the prompt
*   then pass these tokens to the model





In [4]:
prompt = "Write an email apologizing to sarah for tragic gardening mishap.Explain how it happened"

In [5]:
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda") # this gives us a dictionary as output

NameError: name 'tokenizer' is not defined

In [66]:
print(input_ids)

{'input_ids': tensor([[     2,   6974,    614,   4553, 236576,    531,  15949,    957,    573,
          44040,  49115, 200941, 236761, 155122,   1217,    625,   8432]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


##we can visualize these input token id's using decode method

In [67]:
print(tokenizer.decode(input_ids['input_ids']))

['<bos>Write an email apologizing to sarah for tragic gardening mishap.Explain how it happened']


##The output of tokenizer is a dictionary like object(a Batchencoding object) that contains key like 'input_ids' and 'attention_mask'.


*   Bute the generate() function takes only tensors as input, therefore we need to provide the inputs this way.
```python
*   model.generate(
    input_ids=input_ids['input_ids'],
    attention_mask=input_ids['attention_mask'],
    max_new_tokens=20
)
```


*   A cleaner way is using the ** operator




    



In [71]:
mail_output = model.generate(**input_ids, max_new_tokens = 100)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


In [72]:
print(mail_output)

tensor([[     2,   6974,    614,   4553, 236576,    531,  15949,    957,    573,
          44040,  49115, 200941, 236761, 155122,   1217,    625,   8432, 236761,
            107,  31163,  21516, 236764,    108, 236777,   1006,   5712,    531,
          64853,  57428,    573,    506,  44040,  49115, 200941,    600,   8432,
            580,    870,   3682,   1619,   3551,  35980,   6915,   3121,   6517,
            528,    506,   2297, 236761,    564,    691,   2844,    580,  49115,
           5699,    657,   1041,   2033,    532,    691,   1401,  25269, 236761,
            564,   1006,    834,  15025,    573,    506,   3967,    529,   1116,
           6571,    532,    506,  33011,    564,   7321,    531,   1116, 236761,
            108, 236777,    795,    776,   1041,   1791,    531,   1386,    872,
            573,    506,   3068,   1651,    564,    740, 236761,    564,   1006,
           6995,    580,   3978,    496,    861,  28516,    532,   8009,   1070,
            861,   6485,    

##we can see that now the output generated is a bunch of tokenid's.


*   Therefore we need to convert them again into tokens using **decode** method


In [85]:
response = tokenizer.decode(mail_output) #This will give a list containing a string, which is our output

In [86]:
response

['<bos>Write an email apologizing to sarah for tragic gardening mishap.Explain how it happened.\nDear Sarah,\n\nI am writing to sincerely apologize for the tragic gardening mishap that happened on [Date]. My grandmother passed away earlier in the month. I was working on gardening projects at my home and was very upset. I am so sorry for the loss of her memory and the distress I caused to her.\n\nI will do my best to make up for the past while I can. I am planning on getting a new shed and adding some new plants to my garden, and I will']

In [88]:
response[0] # the markdown function need only text not list so we give it this response[0]

'<bos>Write an email apologizing to sarah for tragic gardening mishap.Explain how it happened.\nDear Sarah,\n\nI am writing to sincerely apologize for the tragic gardening mishap that happened on [Date]. My grandmother passed away earlier in the month. I was working on gardening projects at my home and was very upset. I am so sorry for the loss of her memory and the distress I caused to her.\n\nI will do my best to make up for the past while I can. I am planning on getting a new shed and adding some new plants to my garden, and I will'

##If we want to display the output in a better way we use python library *IPython.display*

In [89]:
from IPython.display import Markdown, display
display(Markdown(response[0]))

<bos>Write an email apologizing to sarah for tragic gardening mishap.Explain how it happened.
Dear Sarah,

I am writing to sincerely apologize for the tragic gardening mishap that happened on [Date]. My grandmother passed away earlier in the month. I was working on gardening projects at my home and was very upset. I am so sorry for the loss of her memory and the distress I caused to her.

I will do my best to make up for the past while I can. I am planning on getting a new shed and adding some new plants to my garden, and I will

##Now we check the same thing for PHI3 mini

In [6]:
phi3_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-3-mini-4k-instruct",
                                                  device_map = "auto",
                                                  torch_dtype = "auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
phi3_tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [ ]:
input_ids = phi3_tokenizer(prompt, return_tensors="pt").to("cuda")

In [8]:
print(input_ids)

NameError: name 'input_ids' is not defined